# 04 — Модель 3: ruBERT-tiny2

**`cointegrated/rubert-tiny2`**

Дистиллированная компактная модель: в 15× меньше ruBERT-base, при этом сохраняет ~85% качества. Обучается быстро, полезна как лёгкая baseline для ансамблирования.

In [ ]:
import sys, os, json

PROJECT_ROOT = "/kaggle/input/sentiment-analysis" if os.path.exists("/kaggle") else os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "../results"
os.makedirs(f'{WORKING_DIR}/models/rubert_tiny', exist_ok=True)

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from datasets import load_from_disk
import warnings
warnings.filterwarnings('ignore')

with open(f'{WORKING_DIR}/label_names.json') as f:
    label_names = json.load(f)

DATA_PATH = f'{WORKING_DIR}/processed_data'
if os.path.exists(DATA_PATH):
    dataset = load_from_disk(DATA_PATH)
else:
    from src.data_loader import load_data
    from src.preprocessor import preprocess_batch
    dataset, label_names, _ = load_data()
    dataset = dataset.map(lambda b: preprocess_batch(b, clean=True), batched=True)

for split in dataset:
    print(f'  {split}: {len(dataset[split]):,}')

## Обучение

In [ ]:
from src.trainer import train_model

model, tokenizer, results = train_model(
    model_name='cointegrated/rubert-tiny2',
    dataset=dataset,
    output_dir=f'{WORKING_DIR}/models/rubert_tiny',
    num_labels=len(label_names),
    label_names=label_names,
    num_epochs=5,
    batch_size=64,           # Маленькая модель — крупный батч
    learning_rate=3e-5,
    max_length=128,
    fp16=True,
    seed=42,
)

## Результаты

In [ ]:
from src.evaluation import evaluate_predictions, plot_confusion_matrix
import numpy as np

preds  = np.array(results['predictions'])
labels = np.array(results['true_labels'])
probs  = np.array(results['probabilities'])

metrics = evaluate_predictions(labels, preds, probs, model_name='ruBERT-tiny2', label_names=label_names)
for k, v in metrics.items():
    if k != 'model':
        print(f'  {k:<25s}: {v:.4f}')

In [ ]:
plot_confusion_matrix(
    labels, preds,
    model_name='ruBERT-tiny2',
    label_names=label_names,
    save_path=f'{WORKING_DIR}/cm_rubert_tiny.png',
)
print(f'F1-macro = {metrics["f1_macro"]:.4f}')